In [1]:
%load_ext autoreload
%autoreload 2

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import matplotlib as mpl

from tqdm.auto import tqdm

import warnings
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

from multiprocess import Pool

from scipy import stats


In [3]:
workspace_root = os.getcwd()
sys.path.insert(0, workspace_root + "/../../../")
import pyanalib.pandas_helpers as ph
from makedf.util import *

sys.path.insert(0, workspace_root + "/../")
import kinematics
import gump_cuts as gc
import loaddf
import syst
import importlib

In [4]:
PLOTDIR = "/exp/sbnd/app/users/nrowe/cafpyana/analysis_village/gump/nb/Neutrino26Plots"

DOSAVE = False

import os
os.makedirs(PLOTDIR, exist_ok=True)
os.makedirs(PLOTDIR + "/png", exist_ok=True)
os.makedirs(PLOTDIR + "/pdf", exist_ok=True)

In [5]:
DETECTOR = "ICARUS Run4"

In [6]:
INCLUDE_DIRT=True
# Just a big list of files, needs to be updated
DF_DIR = "/exp/sbnd/data/users/gputnam/GUMP/sbn-rewgted-7/"

if DETECTOR == "ICARUS Run2":
    ONBEAM = DF_DIR + "ICARUS_SpringRun2BNB_unblind.df"
    OFFBEAM = [DF_DIR + "ICARUS_SpringRun2BNBOff_unblind.df"]
    
    ONBEAMPOT = DF_DIR + "ICARUS_SpringRun2BNB_unblind_prescaled_POT.df"

    MC_FILES = [DF_DIR + "ICARUS_SpringMCOverlay_rewgt.df"]

    DIRT_FILE = DF_DIR + "ICARUS_Spring_Overlay_Dirt.df"
    
    DETVAR_FILES = []
    DETVAR_NAMES = []
elif DETECTOR == "ICARUS Run4":
    ONBEAM = DF_DIR + "ICARUS_SpringRun4BNB_unblind_prescaled.df"
    OFFBEAM = [DF_DIR + "ICARUS_SpringRun4BNBOff_unblind_prescaled.df"]
    
    ONBEAMPOT = DF_DIR + "ICARUS_SpringRun4BNB_unblind_prescaled_POT.df"

    MC_FILES = [DF_DIR + "ICARUSRun4_SpringMCOverlay_rewgt_%i.df" % i for i in range(5)]

    DIRT_FILE = DF_DIR + "ICARUSRun4_Spring_Overlay_Dirt_lowE.df"
    
    DETVAR_FILES = []
    DETVAR_NAMES = []
elif DETECTOR == "SBND": 
    ONBEAM = DF_DIR + "SBND_SpringBNBData_FixedDev.df"
    OFFBEAM = DF_DIR + "SBND_SpringBNBOffData_5000.df"

    MC_FILES = [DF_DIR + "SBND_SpringMC_rewgt_E_%i.df" % i for i in range(10)]
    DIRT_FILE = DF_DIR + "SBND_SpringLowEMC.df"

    DETVAR_FILES = [
        DF_DIR + "SBND_SpringMC_Nom.df",
        DF_DIR + "SBND_SpringMC_WMXThetaXW.df",
        DF_DIR + "SBND_SpringMC_WMYZ.df",
        DF_DIR + "SBND_SpringMC_2xSCE.df",
    ]
    DETVAR_NAMES = ["Nominal",
                    "WM $X\\theta_{xw}$", "WM $YZ$", 
                    "2x SCE"]

In [7]:
import h5py

def read_dfs(file, key):
    with h5py.File(file, "r") as f:
        keys = [k for k in f.keys() if k.startswith(key)]
        return pd.concat([pd.read_hdf(file, k) for k in keys])

In [8]:
# get gate based scale factor for offbeam vs onbeam

if "ICARUS" in DETECTOR:
    ngates_ON = read_dfs(ONBEAM, "trig").gate_delta.sum()*(1-1/100.)
    ngates_OFF = sum([read_dfs(O, "trig").gate_delta.sum()*(1-1/20.) for O in OFFBEAM])
    
    OFF_w = ngates_ON / ngates_OFF
elif "SBND" in DETECTOR:
    ngates_ON = read_dfs(ONBEAM, "bnb").shape[0]
    ngates_OFF = read_dfs(OFFBEAM, "hdr").noffbeambnb.sum()

    f_factor = 0.0754 # fraction of gates triggered by a neutrino from SBN-doc-41013-v4
    OFF_w = (1. - f_factor) * (ngates_ON) / (ngates_OFF)

In [9]:
if "ICARUS" in DETECTOR:
    # POT = pd.read_hdf(ONBEAMPOT).pot.sum()*1e12
    POT = read_dfs(ONBEAM, "hdr").merge(pd.read_hdf(ONBEAMPOT), left_index=True, right_index=True, how="left").pot_y.sum()*1e12
elif "SBND" in DETECTOR:
    POT = read_dfs(ONBEAM, "bnb").TOR875.sum()

In [10]:
def FV(df):    
    det = DETECTOR.split(" ")[0]
    ret = gc.slcfv_cut(df, det) 
    return ret

def flash_cut(df):    
    det = DETECTOR.split(" ")[0]
    ret = gc.flash_cut(df, det)
    return ret

def containment_cut(df):    
    det = DETECTOR.split(" ")[0]
    ret = gc.mufv_cut(df, det) & gc.pfv_cut(df, det) 
    return ret

def pid_cut(df):    
    ret = gc.pid_cut(df.mu_chi2_of_mu_cand, df.mu_chi2_of_prot_cand,
                  df.prot_chi2_of_mu_cand, df.prot_chi2_of_prot_cand,
                  df.mu_len)
    return ret

In [11]:
cols_to_drop = ['is_clear_cosmic', 'crlongtrkdiry', 'p_len', 'mu_E', 'mu_T', 'p_E', 'p_T', 'del_Tp', 'del_phi', 'has_stub',
                'true_pcand_pdg', 'true_p_dir_x', 'true_p_dir_y', 'true_p_dir_z', 'true_pcand_dir_x', 'true_pcand_dir_y', 'true_pcand_dir_z', 'true_pcand_end_x', 'true_pcand_end_y', 'true_pcand_end_z',
                'true_mucand_pdg', 'true_mu_dir_x', 'true_mu_dir_y', 'true_mu_dir_z', 'true_mucand_dir_x', 'true_mucand_dir_y', 'true_mucand_dir_z', 'true_mucand_end_x', 'true_mucand_end_y', 'true_mucand_end_z',
                'stub_l0_5cm_dedx','stub_l0_5cm_charge','stub_l1cm_dedx','stub_l1cm_charge','stub_l2cm_dedx','stub_l2cm_charge','stub_l3cm_dedx','stub_l3cm_charge','stub_l4cm_dedx','stub_l4cm_charge',
                'prot_chi2smear5_of_prot_cand', 'prot_chi2smear5_of_mu_cand', 'mu_chi2smear5_of_mu_cand', 'mu_chi2smear5_of_prot_cand', 'tmatch_pur', 'tmatch_eff', 'true_baseline', 'true_nu_pdg_x', 'true_nu_pdg_y',
                'true_nmu_27MeV', 'true_np_20MeV', 'true_np_50MeV', 'true_npi_30MeV', 'is_cosmic', 'flash_sumpe', 'true_mucand_p', 'true_pcand_p', 'mu_true_p', 'p_true_p', 'true_mu_end_x', 
                'true_p_end_x', 'true_mu_end_y', 'true_p_end_y', 'true_mu_end_z', 'true_p_end_z','crthit', 'true_nu_E', 'nu_E0', 'p_true_pdg', 'mu_true_pdg', 'mu_chi22lo_of_mu_cand', 'mu_chi22hi_of_mu_cand', 'prot_chi22lo_of_mu_cand', 'prot_chi22hi_of_mu_cand',
                'mu_chi22lo_of_prot_cand', 'mu_chi22hi_of_prot_cand', 'prot_chi22lo_of_prot_cand', 'prot_chi22hi_of_prot_cand', 'true_mu_p', 'true_p_p', 'pot_univ']

In [12]:
# LOAD MC CV SAMPLES
if DETECTOR == "SBND":
    ncores=10
else:
    ncores = 3
df, match, mcpot = loaddf.loadl(MC_FILES, njob=min(len(MC_FILES), ncores), include_syst=True, reweight_aFF=True, preselection=FV, detector=DETECTOR, drops=cols_to_drop, progress=False, mem32=True)

[ICARUSRun4_SpringMCOverlay_rewgt_0.df idf=0] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun4_SpringMCOverlay_rewgt_1.df idf=0] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun4_SpringMCOverlay_rewgt_2.df idf=0] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun4_SpringMCOverlay_rewgt_1.df idf=1] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun4_SpringMCOverlay_rewgt_0.df idf=1] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun4_SpringMCOverlay_rewgt_2.df idf=1] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun4_SpringMCOverlay_rewgt_1.df idf=2] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun4_SpringMCOverlay_rewgt_2.df idf=2] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
[ICARUSRun4_SpringMCOverlay_rewgt_0.df idf=2] dedup: dropped 0 duplicate

In [17]:
memory_per_col = df.memory_usage(deep=True) / (1024 ** 2)
print(memory_per_col.sort_values(ascending=False).head(20))

detector                     1120.117573
true_isfv                     571.461651
true_isothernumucc            570.101543
true_issig                    569.761730
true_isnc                     569.751724
Run                           142.237152
prot_chi2_of_prot_cand         71.118576
slc_vtx_x                      71.118576
slc_vtx_y                      71.118576
slc_vtx_z                      71.118576
mu_chi2lo_of_prot_cand         71.118576
mu_chi2lo_of_mu_cand           71.118576
mu_chi2hi_of_mu_cand           71.118576
mu_chi2smear13_of_mu_cand      71.118576
nu_score                       71.118576
mu_chi2_of_mu_cand             71.118576
mu_chi2_of_prot_cand           71.118576
prot_chi2_of_mu_cand           71.118576
mu_dir_z                       71.118576
prot_chi2lo_of_mu_cand         71.118576
dtype: float64


In [13]:
if all(col in df.columns for col in cols_to_drop):
    df.drop(columns=cols_to_drop, inplace=True)
df[df.select_dtypes(include=['float64']).columns] = df.select_dtypes(include=['float64']).astype('float32')

In [20]:
import sys
def sizeof_fmt(num, suffix='B'):
    ''' by Fred Cirera,  https://stackoverflow.com/a/1094933/1870254, modified'''
    for unit in ['','Ki','Mi','Gi','Ti','Pi','Ei','Zi']:
        if abs(num) < 1024.0:
            return "%3.1f %s%s" % (num, unit, suffix)
        num /= 1024.0
    return "%.1f %s%s" % (num, 'Yi', suffix)

for name, size in sorted(((name, sys.getsizeof(value)) for name, value in list(
                          locals().items())), key= lambda x: -x[1])[:10]:
    print("{:>30}: {:>8}".format(name, sizeof_fmt(size)))

                            df: 19.7 GiB
                          dirt: 112.0 MiB
                         match: 54.3 MiB
                     dirtmatch:  1.1 MiB
                          _i11:  1.6 KiB
                           _i6:  1.6 KiB
                            _i:  1.2 KiB
                          _i19:  1.2 KiB
              InteractiveShell:  1.0 KiB
                          tqdm:  1.0 KiB


In [21]:
import ctypes
import gc as garbc

def hard_reclaim():
    garbc.collect()
    try:
        # This explicitly tells the C library to release memory
        ctypes.CDLL('libc.so.6').malloc_trim(0)
        print("Malloc trim successful.")
    except Exception as e:
        print(f"Failed to trim: {e}")

hard_reclaim()

Malloc trim successful.


In [1]:
# LOAD DETECTOR VARIATION SAMPLES

if len(DETVAR_FILES) > 0:
    detvars, detvars_match, detvar_pots = zip(*[loaddf.load(f, include_syst=False, detector=DETECTOR, drops=cols_to_drop, progress=False, mem32=True) for f in tqdm(DETVAR_FILES)])
    detvars, detvar_pots = loaddf.match_common_evts(detvars_match, detvars, detvar_pots)
else:
    detvars = []
    detvar_pots = []

NameError: name 'DETVAR_FILES' is not defined

In [23]:
if 'pot_univ' in cols_to_drop:
    cols_to_drop.remove('pot_univ')

for df_i in range(len(detvars)):
    try:
        detvars[df_i].drop(columns=cols_to_drop, inplace=True)
        detvars[df_i][detvars[df_i].select_dtypes(include=['float64']).columns] = detvars[df_i].select_dtypes(include=['float64']).astype('float32')
    except:
        print("failed, prob already removed")

In [24]:
loaddf.scale_pot(df, mcpot, POT)
for i in range(len(detvars)):
    print(DETVAR_NAMES[i])
    loaddf.scale_pot(detvars[i], detvar_pots[i], POT)

(np.float32(1.05021804e+21), np.float64(0.00851648909301846))

In [25]:
# This removes things in the AV, so that our dirt sample is actually a dirt sample
def ICARUS_dirtcut(df):
    # mbox from CV sample
    xlo = -378.49
    ylo = -191.86
    zlo = -904.950652270838

    xhi = 378.49
    yhi = 144.96
    zhi = 904.950652270838
    vtx = pd.DataFrame({'x': df.true_vtx_x,
                       'y': df.true_vtx_y,
                       'z': df.true_vtx_z}, index=df.index)

    return ~((vtx.x > xlo) & (vtx.x < xhi) & (vtx.y > ylo) & (vtx.y < yhi) & (vtx.z > zlo) & (vtx.z < zhi))

if INCLUDE_DIRT and DIRT_FILE is not None:
    dirt, dirtmatch, dirtpot = loaddf.load(DIRT_FILE, include_syst=False, preselection=FV, detector=DETECTOR, mem32=True, drops=cols_to_drop)
    
    if all(col in dirt.columns for col in cols_to_drop):
        dirt.drop(columns=cols_to_drop, inplace=True)
    dirt[dirt.select_dtypes(include=['float64']).columns] = dirt.select_dtypes(include=['float64']).astype('float32')
        
    loaddf.scale_pot(dirt, dirtpot, POT)
    
    if DETECTOR == "ICARUS":
        dirt = dirt[ICARUS_dirtcut(dirt)]

    dirt["crthit"] = False
    dirt["dirt"] = True
    df["dirt"] = False
elif INCLUDE_DIRT:
    df["dirt"] = False

[ICARUSRun4_Spring_Overlay_Dirt_lowE.df idf=0] dedup: dropped 0 duplicated ('run', 'evt', 'nu_E0') keys (0 hdr rows)
load ret


(np.float32(3.1250896e+19), np.float64(0.28620525305138445))

In [ ]:
if INCLUDE_DIRT and DIRT_FILE is not None:
    # reset df if needed
    df = df[~df.dirt]
    
    # add dirt to the CV, and also include it (below) as a 100 syst unc.
    df = pd.concat([df, dirt])

    # keep copy of non-dirt around for chi2 systematic variation CV
    df0 = df.loc[~df.dirt, [c for c in df.columns if "univ" not in c]]
    
    # disable systematic weights associated with dirt events
    for c in df.columns:
        if "univ" in c:
            df[c] = df[c].fillna(1)
            
    # also add it to the detector variations, for consistency
    for i in range(len(detvars)):
        detvars[i] = pd.concat([detvars[i], dirt])
elif INCLUDE_DIRT:
    # keep copy of non-dirt around for chi2 systematic variation CV
    df0 = df.loc[~df.dirt, [c for c in df.columns if "univ" not in c]]
del dirt

In [ ]:
# ADD IN PID VARIATIONS
def v_variation(df, setvars):
    df = df[[c for c in df.columns if "univ" not in c]].copy()
    for (new, old) in setvars:
        df[new] = df[old]
    return df

def v_chi2smear(df):
    setvars = [
        ("mu_chi2_of_mu_cand", "mu_chi2smear13_of_mu_cand"),
        ("mu_chi2_of_prot_cand",  "mu_chi2smear13_of_prot_cand"),
        ("prot_chi2_of_mu_cand", "prot_chi2smear13_of_mu_cand"),
        ("prot_chi2_of_prot_cand",  "prot_chi2smear13_of_prot_cand"),
    ]
    return v_variation(df, setvars)


def v_chi2hi(df):
    setvars = [
        ("mu_chi2_of_mu_cand", "mu_chi2hi_of_mu_cand"),
        ("mu_chi2_of_prot_cand",  "mu_chi2hi_of_prot_cand"),
        ("prot_chi2_of_mu_cand", "prot_chi2hi_of_mu_cand"),
        ("prot_chi2_of_prot_cand",  "prot_chi2hi_of_prot_cand"),
    ]
    return v_variation(df, setvars)

def v_chi2lo(df):
    setvars = [
        ("mu_chi2_of_mu_cand", "mu_chi2lo_of_mu_cand"),
        ("mu_chi2_of_prot_cand",  "mu_chi2lo_of_prot_cand"),
        ("prot_chi2_of_mu_cand", "prot_chi2lo_of_mu_cand"),
        ("prot_chi2_of_prot_cand",  "prot_chi2lo_of_prot_cand"),
    ]
    return v_variation(df, setvars)

def v_chi22hi(df):
    setvars = [
        ("mu_chi2_of_mu_cand", "mu_chi22hi_of_mu_cand"),
        ("mu_chi2_of_prot_cand",  "mu_chi22hi_of_prot_cand"),
        ("prot_chi2_of_mu_cand", "prot_chi22hi_of_mu_cand"),
        ("prot_chi2_of_prot_cand",  "prot_chi22hi_of_prot_cand"),
    ]
    return v_variation(df, setvars)

def v_chi22lo(df):
    setvars = [
        ("mu_chi2_of_mu_cand", "mu_chi22lo_of_mu_cand"),
        ("mu_chi2_of_prot_cand",  "mu_chi22lo_of_prot_cand"),
        ("prot_chi2_of_mu_cand", "prot_chi22lo_of_mu_cand"),
        ("prot_chi2_of_prot_cand",  "prot_chi22lo_of_prot_cand"),
    ]
    return v_variation(df, setvars)

In [ ]:
df0 = df0.drop(columns=[col for col in df0.columns if 'univ' in col])

sys.modules['IPython'].get_ipython().user_ns['Out'] = {}

# Specifically clear the last-result references
_ = None
__ = None
___ = None

# Force the garbage collector to see the newly freed references
import gc
gc.collect()

def hard_reclaim():
    gc.collect()
    try:
        # This explicitly tells the C library to release memory
        ctypes.CDLL('libc.so.6').malloc_trim(0)
        print("Malloc trim successful.")
    except Exception as e:
        print(f"Failed to trim: {e}")

hard_reclaim()

In [ ]:
chi2_detvars = [[v_chi2hi(df0), v_chi2lo(df0)], v_chi2smear(df0)]
#chi2_2x_detvars = [[v_chi22hi(df0), v_chi22lo(df0)], v_chi2smear(df0)]

In [ ]:
import gump_cuts as gc

def TrueAV(df, det):
    if det == "ICARUS Run4":
        df['Run'] = 4
    elif det == "ICARUS Run2":
        df['Run'] = 2    
    elif det == "SBND":
        df['Run'] = 1
    else:
        print('unclear run!!!')
        
    vtx = pd.DataFrame({'Run': df.Run,
                       'x': df.true_vtx_x,
                       'y': df.true_vtx_y,
                       'z': df.true_vtx_z}, index=df.index)
    return gc._fv_cut(vtx, det, 0, 0, 0, 0)

def OOAV(df):
    det = DETECTOR.split(" ")[0]
    return ~np.isnan(df.true_vtx_x) & ~TrueAV(df, det)


df_ooav = df[OOAV(df)].copy()

In [ ]:
ONdf,_,_ = loaddf.load(ONBEAM, load_truth=False, include_syst=False, detector=DETECTOR, preselection=FV, match_Enu=False)
if "ICARUS" in DETECTOR:
    OFFdf,_,_ = loaddf.loadl(OFFBEAM, load_truth=False, include_syst=False, detector=DETECTOR, preselection=FV, match_Enu=False)
else:
    OFFdf,_,_ = loaddf.load(OFFBEAM, load_truth=False, include_syst=False, detector=DETECTOR, preselection=FV, match_Enu=False)

In [ ]:
mode_list = [1, 10, 0]

mode_labels = ["Cosmic", "Dirt", 'Other $\\nu$', '$\\nu_\\mu$ CC RES', '$\\nu_\\mu$ CC MEC', '$\\nu_\\mu$ CC QE']
mode_colors = ["#95af8b", "#43140b", "#c89648", "#1e3f54", "#d54c28", "#315031"]

def nuFV(df):
    det = DETECTOR.split(" ")[0]

    vtx = pd.DataFrame({
        "Run": df.Run,
        "x": df.true_vtx_x,
        "y": df.true_vtx_y,
        "z": df.true_vtx_z,
    })
    return ~np.isnan(df.true_vtx_x) & gc._fv_cut(vtx, det, 0, 0, 0, 0)

def breakdown_mode(var, df):
    """Break down variable by interaction mode."""
    numu_cc = (np.abs(df.true_pdg) == 14) & (df.true_isnc == False)
    fid = nuFV(df)
    fid = ~df.dirt

    ret = [
        var[np.isnan(df.genie_mode)],
        var[~fid & ~np.isnan(df.genie_mode)],
        var[(~np.any([df.genie_mode == i for i in mode_list], axis=0) | ~numu_cc) & fid & ~np.isnan(df.genie_mode)]
    ] +\
        [var[(df.genie_mode == i) & numu_cc & fid] for i in mode_list]
        
    return ret

In [ ]:
FONTSIZE = 14
HAWKS_COLORS = ["#315031", "#d54c28", "#1e3f54", "#c89648", "#43140b", "#95af8b"]

def add_style(ax, xlabel, title="", det="ICARUS", ylabel="Area Normalized", legend_loc="upper right", legend_ncol=3):
    ax.tick_params(axis='both', which='both', direction='in', length=6, width=1.5, labelsize=FONTSIZE, top=True, right=True)
    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
    ax.set_xlabel(xlabel, fontsize=FONTSIZE, fontweight='bold')
    ax.set_ylabel(ylabel, fontsize=FONTSIZE, fontweight='bold')
    ax.set_title(f"$\\bf{{{det}}}$  {title}", fontsize=FONTSIZE+2)
    ax.legend(fontsize=FONTSIZE, loc=legend_loc, ncol=legend_ncol)


In [ ]:
def f_chi2(NMC, Ndata, cov):
    # ignore singular entries
    which_bin = NMC > 0

    NMC = NMC[which_bin]
    Ndata = Ndata[which_bin]
    cov = cov[which_bin, :]
    cov = cov[:, which_bin]

    delta = NMC - Ndata
    try:
        cov_inv = np.linalg.inv(cov)
    except np.linalg.LinAlgError as _:
        return -1, which_bin.sum()
        
    return delta@cov_inv@delta, which_bin.sum()

In [ ]:
if DETECTOR == "SBND":
    x_range = np.linspace(-200, 200, 21)
    y_range = np.linspace(-200, 200, 21)
    z_range = np.linspace(0, 500, 21)
else:
    x_range = np.linspace(-360, 360, 21)
    y_range = np.linspace(-182, 135, 21)
    z_range = np.linspace(-895, 895, 21)

In [ ]:
extravarnames = [
    "mu_costh",
    "mu_phi",
    "p_costh",
    "p_phi",
]

extravars = [
    lambda d: np.cos(d.mu_dir_z),
    lambda d: np.arctan2(d.mu_dir_x, d.mu_dir_y)*180/np.pi,
    lambda d: np.cos(d.p_dir_z),
    lambda d: np.arctan2(d.p_dir_x, d.p_dir_y)*180/np.pi,
]

mconly_extravarnames = [
    "Eres"
]

mconly_extravars = [
    lambda d: (d.nu_E_calo - d.nu_E_true)/d.nu_E_true
]

In [ ]:
for v, vname in zip(mconly_extravars, mconly_extravarnames):
    df[vname] = v(df)
    df0[vname] = v(df0)
    df_ooav[vname] = v(df_ooav)

    #if INCLUDE_DIRT and DIRT_FILE is not None:
    #    dirt[vname] = v(dirt)

    for i in range(len(detvars)):
        detvars[i][vname] = v(detvars[i])

    for i in range(len(chi2_detvars)):
        if isinstance(chi2_detvars[i], list):
            for j in range(len(chi2_detvars[i])):
                chi2_detvars[i][j][vname] = v(chi2_detvars[i][j])
        else:
            chi2_detvars[i][vname] = v(chi2_detvars[i])

In [ ]:
for v, vname in zip(extravars, extravarnames):
    df[vname] = v(df)
    df0[vname] = v(df0)
    df_ooav[vname] = v(df_ooav)
    ONdf[vname] = v(ONdf)
    OFFdf[vname] = v(OFFdf)

    #if INCLUDE_DIRT and DIRT_FILE is not None:
    #    dirt[vname] = v(dirt)

    for i in range(len(detvars)):
        detvars[i][vname] = v(detvars[i])

    for i in range(len(chi2_detvars)):
        if isinstance(chi2_detvars[i], list):
            for j in range(len(chi2_detvars[i])):
                chi2_detvars[i][j][vname] = v(chi2_detvars[i][j])
        else:
            chi2_detvars[i][vname] = v(chi2_detvars[i])

In [ ]:
df = gc.add_opening_angle_mu_p(df)
df0 = gc.add_opening_angle_mu_p(df0)
df_ooav = gc.add_opening_angle_mu_p(df_ooav)
ONdf = gc.add_opening_angle_mu_p(ONdf)
OFFdf = gc.add_opening_angle_mu_p(OFFdf)

In [ ]:
def step_0(df):
    return (df.nu_E_calo > -999.)

def step_1(df):
    return step_0(df) & FV(df)

def step_2(df):
    return step_1(df) & flash_cut(df)
    
def step_3(df):
    return step_2(df) & gc.cosmic_cut(df)

def step_4(df):
    return step_3(df) & gc.twoprong_cut(df)
    
def step_5(df):
    return step_4(df) & containment_cut(df)

def step_6(df):
    return step_5(df) & pid_cut(df)

In [ ]:
cuts = [
    step_6
]

cutnames = [
    "GUMP Signal Box"
]

plotvars = [
    ["del_p"]
]

labels = [
    ["$\\delta p$ [GeV]"]
]

bins = [
    [np.linspace(0., 1., 21)]
]

for c, cname in zip(cuts, cutnames):
    df[cname] = c(df)
    df0[cname] = c(df0)
    df_ooav[cname] = c(df_ooav)
    ONdf[cname] = c(ONdf)
    OFFdf[cname] = c(OFFdf)
    #dirt[cname] = c(dirt)
    
    for i in range(len(detvars)):
        detvars[i][cname] = c(detvars[i])
        
    for i in range(len(chi2_detvars)):
        if isinstance(chi2_detvars[i], list):
            for j in range(len(chi2_detvars[i])):
                chi2_detvars[i][j][cname] = c(chi2_detvars[i][j])
        else:
            chi2_detvars[i][cname] = c(chi2_detvars[i])
            
    # for i in range(len(chi2_2x_detvars)):
    #     if isinstance(chi2_2x_detvars[i], list):
    #         for j in range(len(chi2_2x_detvars[i])):
    #             chi2_2x_detvars[i][j][cname] = c(chi2_2x_detvars[i][j])
    #     else:
    #         chi2_2x_detvars[i][cname] = c(chi2_2x_detvars[i])

In [ ]:
#pot_scales = {'ps1': 0.5, 'ms1': -0.525805, 'ps2': 1.0, 'ms2': -1.12726, 'ps3': 1.5, 'ms3': -1.7286, 'cv': 0.0}

# Define systematic uncertainties
if "ICARUS" in DETECTOR:
    systematics = syst.SystematicList([
        loaddf.FluxSystematic(df),
        loaddf.XSecSystematic(df),
        syst.NormalizationSystematic(0.02),
        syst.SystematicList(
            [syst.SampleSystematic(d, cvdf=df0) for d in chi2_detvars]),
        syst.SystSampleSystematic(df_ooav)])
else:
    systematics = syst.SystematicList([
        loaddf.FluxSystematic(df),
        loaddf.XSecSystematic(df),
        syst.NormalizationSystematic(0.02),
        syst.SystematicList(
            [syst.SampleSystematic(d, cvdf=detvars[0]) for d in detvars[1:]] +
            [syst.SampleSystematic(d, cvdf=df0) for d in chi2_detvars]),
        syst.SystSampleSystematic(df_ooav)])

In [ ]:
def make_plot_data(var, bins, cut, mc_weight, breakdown, areanorm, breakdown_labels, breakdown_colors, xlabel, title, 
                   det="ICARUS", fillna=np.nan, syst=systematics):
    
    pvars = breakdown(df.loc[df[cut], var].fillna(fillna), df[df[cut]])
    weights = breakdown(df.loc[df[cut], mc_weight], df[df[cut]])
    NMC_breakdown = []
    for pvar, w in zip(pvars, weights):    
        thisNMC, bins = np.histogram(pvar, bins=bins, weights=w)
        NMC_breakdown.append(thisNMC)
        
    NMC,_ = np.histogram(df.loc[df[cut], var].fillna(fillna), bins=bins, weights=df.loc[df[cut], mc_weight])
    NMC_abs = NMC
    if areanorm:
        diff = (bins[1:] - bins[:-1])
        norm = np.sum(NMC*diff)
        if norm > 1e-5:
            NMC = NMC / norm
            for i in range(len(NMC_breakdown)):
                NMC_breakdown[i] = NMC_breakdown[i] / norm

    NMC_breakdown = np.array(NMC_breakdown)
    NON,_ = np.histogram(ONdf.loc[ONdf[cut], var].fillna(fillna), bins=bins)
    NOff,_ = np.histogram(OFFdf.loc[OFFdf[cut], var].fillna(fillna), bins=bins)

    N = NON - NOff*OFF_w
    Nerr = np.sqrt(NON + NOff*OFF_w**2)
    if areanorm:
        diff = (bins[1:] - bins[:-1])
        
        norm = np.sum(N*diff)
        if norm > 1e-5:
            N = N / norm
            Nerr = Nerr / norm
    
    cov = syst.cov(var, cut, bins, NMC_abs, shapeonly=areanorm, fillna=fillna)
    err = np.sqrt(np.diag(cov))

    cov_w_stat = cov + np.diag(Nerr**2) # add stat uncertainty
    chi2, ndof = f_chi2(NMC, N, cov_w_stat)

    return {
        "det": det,
        "title": title,
        "xlabel": xlabel,
        "bins": bins,
        "areanorm": areanorm,
        "breakdown_labels": breakdown_labels,
        "breakdown_colors": breakdown_colors,
        "NMC_breakdown": NMC_breakdown,
        "NMC_total": NMC,
        "NData": N,
        "NDataErr": Nerr,
        "cov": cov,
        "cov_w_stat": cov_w_stat,
        "chi2": chi2,
        "ndof": ndof,
        "POT": POT
    }


In [ ]:
def ratio_plot(plt, plotdata):
    # Set the figure-level background
    fig, (ax0, ax1) = plt.subplots(2, 1, height_ratios=[3, 1], sharex=True, figsize=(8, 6))
    
    bg_color = '#525b83'
    fig.set_facecolor(bg_color) # Sets the outer padding background

    # --- [Existing data calculation/plotting logic] ---
    bins = plotdata["bins"]
    centers = (bins[:-1] + bins[1:])/2
    NMC_breakdown = plotdata["NMC_breakdown"]
    fill = np.array([centers for _ in range(NMC_breakdown.shape[0])]).T
    
    ax0.hist(fill, bins=bins, stacked=True, label=plotdata["breakdown_labels"],
             color=plotdata["breakdown_colors"], weights=NMC_breakdown.T)
    
    NData = plotdata["NData"]
    NDataErr = plotdata["NDataErr"]
    line = ax0.errorbar(centers, NData, NDataErr, color="black", linestyle="none", marker=".")
    NMC = plotdata["NMC_total"]
    err = np.sqrt(np.diag(plotdata["cov"]))
    
    # Updated error bands to 'lightgray' for better contrast against slate blue
    ax0.fill_between(bins[:-1], NMC+err, NMC-err, facecolor="lightgray", alpha=0.4, 
                     edgecolor="lightgray", linewidth=0.0, step="post")

    ax0_l0, ax0_hi = ax0.get_ylim()
    ax0.set_ylim([ax0_l0, ax0_hi*1.4])
    
    if "Maximum" in plotdata["xlabel"]:
        ax0.set_yscale("log")
        
    ax1.errorbar(centers, NData/NMC, NDataErr/NMC, color="black", linestyle="none", marker=".")
    ax1.set_ylim([0.5, 1.5])
    ax1.axhline([1], color="red", linestyle="--")
    ax1.fill_between(bins[:-1], 1+err/NMC, 1-err/NMC, facecolor="lightgray", alpha=0.4, 
                     edgecolor="lightgray", linewidth=0.0, step="post")
    
    # --- Updated Styling Block ---
    for ax in [ax0, ax1]:
        ax.set_facecolor(bg_color)
        # Removed set_alpha(0.7) to ensure the hexcode is opaque and accurate
        ax.patch.set_alpha(1.0) 
        
        for spine in ax.spines.values():
            spine.set_edgecolor('white')
            spine.set_linewidth(1.5)

    # White Text and Ticks
    ax0.tick_params(axis='both', colors='white', labelcolor='white', direction='in', 
                    length=6, width=1.5, top=True, right=True)
    ax1.tick_params(axis='both', colors='white', labelcolor='white', direction='in', 
                    length=6, width=1.5, top=True, right=True)
    
    ax1.set_xlabel(plotdata["xlabel"], fontsize=FONTSIZE, fontweight='bold', color='white')
    ylabel_str = 'Area Normalized' if plotdata["areanorm"] else 'Events / %.1f$\\times 10^{19}$ POT'
    ax0.set_ylabel(ylabel_str % (plotdata["POT"]/1e19), fontsize=FONTSIZE, fontweight='bold', color='white')
    
    det = plotdata["det"]
    title = plotdata["title"]
    ax0.set_title(f"$\\bf{{{det}}}$ {title}", fontsize=FONTSIZE+2, color='white')

    # Legend and Annotations
    ld = ax0.legend([line], ["Data\n(ON Beam - OFF)"], frameon=False, loc="upper left", 
                    fontsize=10, labelcolor='white')
    ax0.legend(fontsize=12, loc="upper right", ncol=2, reverse=True, frameon=False, labelcolor='white')
    ax0.add_artist(ld)

    chi2_str = "$\\chi^2_\\mathrm{shape}$" if plotdata["areanorm"] else "$\\chi^2$"
    ax0.text(0.05, 0.8, "%s: %.1f / %i" % (chi2_str, plotdata["chi2"], plotdata["ndof"] - int(plotdata["areanorm"])),
             verticalalignment="top", horizontalalignment="left", fontsize=FONTSIZE-2, 
             transform=ax0.transAxes, color='white')
    
    plt.subplots_adjust(hspace=0.05)
    return fig, ax0, ax1

In [ ]:
for step_cn, step_v, step_b, step_l in zip(cutnames, plotvars, bins, labels):
    if step_cn == "GUMP Signal Box":
        for v, b, l in zip(step_v, step_b, step_l):
            
            print(step_cn)
            dat = make_plot_data(v, b, step_cn, "glob_scale", breakdown_mode, True, mode_labels, 
                                      HAWKS_COLORS, l, step_cn, fillna=-1, det=DETECTOR)
            ratio_plot(plt, dat)
            savename_png = PLOTDIR + "/png/%s_%s_%s.png" % (dat["det"].replace(" ", "-"), step_cn.replace(" ", "").replace(".", "").lower(), v)
            print(savename_png)
            plt.savefig(savename_png, bbox_inches="tight")
            plt.close()

In [ ]:
cuts = [
    step_1,
    step_2,
    step_3,
    step_4,
    step_5,
    step_6
]

cutnames = [
    "FV Cut Step",
    "Flash Cut Step",
    "Cosmic Cut Step",
    "Two Prong Cut Step",
    "Containment Cut Step",
    "PID Cut Step"
]

plotvars = [
    ['Eres'],['Eres'],['Eres'],['Eres'],['Eres'],['Eres']
]

labels = [
    ["($(E_{calo}-E_{true})/E_{true}$)"],
    ["($(E_{calo}-E_{true})/E_{true}$)"],
    ["($(E_{calo}-E_{true})/E_{true}$)"],
    ["($(E_{calo}-E_{true})/E_{true}$)"],
    ["($(E_{calo}-E_{true})/E_{true}$)"],
    ["($(E_{calo}-E_{true})/E_{true}$)"],
]

bins = [
    [np.linspace(-1,1,31)],[np.linspace(-1,1,31)],[np.linspace(-1,1,31)],[np.linspace(-1,1,31)],[np.linspace(-1,1,31)],[np.linspace(-1,1,31)],
]

for c, cname in zip(cuts, cutnames):
    df[cname] = c(df)
    df0[cname] = c(df0)
    df_ooav[cname] = c(df_ooav)
    ONdf[cname] = c(ONdf)
    OFFdf[cname] = c(OFFdf)
    #dirt[cname] = c(dirt)
    
    for i in range(len(detvars)):
        detvars[i][cname] = c(detvars[i])
        
    for i in range(len(chi2_detvars)):
        if isinstance(chi2_detvars[i], list):
            for j in range(len(chi2_detvars[i])):
                chi2_detvars[i][j][cname] = c(chi2_detvars[i][j])
        else:
            chi2_detvars[i][cname] = c(chi2_detvars[i])
            
    # for i in range(len(chi2_2x_detvars)):
    #     if isinstance(chi2_2x_detvars[i], list):
    #         for j in range(len(chi2_2x_detvars[i])):
    #             chi2_2x_detvars[i][j][cname] = c(chi2_2x_detvars[i][j])
    #     else:
    #         chi2_2x_detvars[i][cname] = c(chi2_2x_detvars[i])

In [ ]:
def make_plot_mconly(var, bins, cut, mc_weight, breakdown, areanorm, breakdown_labels, breakdown_colors, xlabel, title, 
                   det="ICARUS", fillna=np.nan, syst=systematics):
    pvars = breakdown(df.loc[df[cut], var].fillna(fillna), df[df[cut]])
    weights = breakdown(df.loc[df[cut], mc_weight], df[df[cut]])
    NMC_breakdown = []
    for pvar, w in zip(pvars, weights):    
        thisNMC, bins = np.histogram(pvar, bins=bins, weights=w)
        NMC_breakdown.append(thisNMC)

    NMC,_ = np.histogram(df.loc[df[cut], var].fillna(fillna), bins=bins, weights=df.loc[df[cut], mc_weight])
    NMC_abs = NMC
    if areanorm:
        diff = (bins[1:] - bins[:-1])
        norm = np.sum(NMC*diff)
        if norm > 1e-5:
            NMC = NMC / norm
            for i in range(len(NMC_breakdown)):
                NMC_breakdown[i] = NMC_breakdown[i] / norm

    NMC_breakdown = np.array(NMC_breakdown)
    
    cov = syst.cov(var, cut, bins, NMC_abs, shapeonly=areanorm, fillna=fillna)
    err = np.sqrt(np.diag(cov))
    
    return {
        "det": det,
        "cut": cut,
        "title": title,
        "xlabel": xlabel,
        "bins": bins,
        "areanorm": areanorm,
        "breakdown_labels": breakdown_labels,
        "breakdown_colors": breakdown_colors,
        "NMC_breakdown": NMC_breakdown,
        "NMC_total": NMC,
        "cov": cov,
        "POT": POT
    }

In [ ]:
FONTSIZE=14
def eres_plot(plt, plotdatas):
    # Set the figure background color
    # Note: 'fig' is required to set the outer background
    fig = plt.gcf()
    bg_color = '#525b83'
    text_color = 'white'
    fig.set_facecolor(bg_color)
    
    ax = plt.gca()
    ax.set_facecolor(bg_color)

    for plotdata, color in zip(plotdatas, HAWKS_COLORS):
        bins = plotdata["bins"]
        centers = (bins[:-1] + bins[1:])/2
    
        NMC = plotdata["NMC_breakdown"][-1]
        plt.hist(centers, bins, weights=NMC, histtype='step', 
                 color=color, label=plotdata['cut'], lw=3)
        
        # Keeping your existing error logic for reference
        err = np.sqrt(np.diag(plotdata["cov"]))

        # Update labels with white text
        plt.xlabel(plotdata["xlabel"], fontsize=FONTSIZE, 
                   fontweight='bold', color=text_color)
        
        det = plotdata["det"]
        plt.title(f"$\\bf{{{det}}}$ $\\nu_\\mu$CCQE Energy Resolution", 
                  fontsize=FONTSIZE+2, color=text_color)
        
        # Update spines and ticks to white
        for spine in ax.spines.values():
            spine.set_linewidth(3)
            spine.set_edgecolor(text_color)
        
        ax.tick_params(colors=text_color, width=2)
        
        plt.ylabel("Area Normalized", fontsize=FONTSIZE, 
                   fontweight='bold', color=text_color)
        plt.ylim(0, 8)
        
        # Legend formatting for dark background
        plt.legend(fontsize=12, loc="upper right", ncol=2, 
                   reverse=True, frameon=True, labelcolor='black', 
                   facecolor='white', framealpha=0.8)
                   
    return plt

# def eres_plot(plt, plotdatas):
#     # --- [Keep your existing data calculation/plotting logic] ---
#     #fig = plt.figure()
#     for plotdata, color in zip(plotdatas, HAWKS_COLORS):
#         bins = plotdata["bins"]
#         centers = (bins[:-1] + bins[1:])/2
    
#         NMC = plotdata["NMC_breakdown"][-1]
#         plt.hist(centers, bins, weights=NMC, histtype='step', color=color, label=plotdata['cut'], lw=3)
#         err = np.sqrt(np.diag(plotdata["cov"]))
#         #plt.fill_between(bins[:-1], NMC+err, NMC-err, facecolor=color, edgecolor=color, linewidth=0.0, step="post", alpha=0.3)
    
#         plt.xlabel(plotdata["xlabel"], fontsize=FONTSIZE, fontweight='bold', color='black')
#         ylabel_str = 'Area Normalized' if plotdata["areanorm"] else 'Events / %.1f$\\times 10^{19}$ POT'
        
    
#         det = plotdata["det"]
#         title = plotdata["title"]
#         plt.title(f"$\\bf{{{det}}}$ $\\nu_\\mu$CCQE Energy Resolution", fontsize=FONTSIZE+2, color='black')
        
#         for spine in plt.gca().spines.values():
#             spine.set_linewidth(3)
#         plt.ylabel("Area Normalized", fontsize=FONTSIZE, fontweight='bold', color='black')
#         plt.ylim(0, 8)
#         plt.legend(fontsize=12, loc="upper right", ncol=2, reverse=True, frameon=True, labelcolor='black')
#     return plt

In [ ]:
step_cn = "GUMP Signal Box"
v = "Eres"
b = np.linspace(-1., 1., 30)
l = "Energy Residual"

dats = []

for step_cn, step_v, step_b, step_l in zip(cutnames, plotvars, bins, labels):
    for v, b, l in zip(step_v, step_b, step_l):
        print(l)
        dat = make_plot_mconly(v, b, step_cn, "glob_scale", breakdown_mode, True, mode_labels, HAWKS_COLORS, l, step_cn, fillna=-999, det=DETECTOR)
        dats.append(dat)

In [ ]:
eres_plot(plt, dats)
if dat["det"].replace(" ", "-") == "ICARUS-Run4" or  dat["det"].replace(" ", "-") == "ICARUS-Run2" or dat["det"].replace(" ", "-") == "ICARUS":
    dat["det"] = "ICARUS"
else:
    dat["det"] = "SBND"

savename_png = PLOTDIR + "/png/ERes%s_%s_%s.png" % (dat["det"].replace(" ", "-"), step_cn.replace(" ", "").replace(".", "").lower(), v)
print(savename_png)
plt.savefig(savename_png, bbox_inches="tight")
plt.close()